# Cart System — Kaggle CPU load test

**Chưa thực thi.** Kaggle là máy phát tải; backend/MySQL/Redis chạy bên ngoài. Chọn Accelerator = None (CPU), kiểm tra Internet và quyền kiểm thử target trước khi chạy. Không tạo tunnel, không mở database/Redis, không đưa JWT signing secret vào notebook. `localhost` ở Kaggle không phải máy Windows.

Đính kèm thư mục `load-testing` từ revision đã duyệt dưới dạng Kaggle Dataset. Giữ password trong Kaggle Secrets với tên `LOAD_TEST_PASSWORD`. Không chia sẻ notebook đã có output nhạy cảm. Các giới hạn môi trường cần xác nhận tại [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys
# Sửa đường dẫn Dataset cho đúng. Thư mục phải có requirements.txt và các script.
SUITE_DIR = Path('/kaggle/input/cart-system-load-testing/load-testing')
WORK = Path('/kaggle/working/cart-load-testing')
if not SUITE_DIR.is_dir():
    raise RuntimeError('Chọn SUITE_DIR đúng với Dataset đã đính kèm')
WORK.mkdir(parents=True, exist_ok=True)
for name in ['requirements.txt', 'prepare.py', 'locustfile.py', 'run.py']:
    shutil.copy2(SUITE_DIR / name, WORK / name)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(WORK / 'requirements.txt')], check=True)


## Cấu hình và quyền truy cập

Nhập origin HTTPS của backend test đã được cho phép. Notebook không deploy ứng dụng. Không dùng địa chỉ MySQL/Redis. Internet bật không bảo đảm target truy cập được: firewall, outbound policy và authentication của proxy vẫn có thể chặn.

Mô tả backend cần ghi máy/CPU/RAM, giới hạn Docker, phiên bản image, MySQL/Redis, vị trí triển khai và revision; để unknown nếu chưa biết, không suy đoán.

In [ ]:
import getpass, ipaddress, json, platform, uuid
import psutil
TARGET_BASE_URL = input('TARGET_BASE_URL: ').strip().rstrip('/')
BACKEND_DESCRIPTION = input('Môi trường backend (hoặc unknown): ').strip() or 'unknown'
from urllib.parse import urlparse
parsed = urlparse(TARGET_BASE_URL)
if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password:
    raise ValueError('Dùng origin HTTPS không chứa credentials')
try:
    if ipaddress.ip_address(parsed.hostname).is_loopback:
        raise ValueError('Loopback Kaggle không trỏ tới backend Windows')
except ValueError as exc:
    if parsed.hostname in ('localhost',) or 'Loopback' in str(exc):
        raise
try:
    from kaggle_secrets import UserSecretsClient
    password = UserSecretsClient().get_secret('LOAD_TEST_PASSWORD')
except (ImportError, RuntimeError):
    password = getpass.getpass('Password tài khoản load test: ')
if not password:
    raise ValueError('Password không được rỗng')
os.environ['TARGET_BASE_URL'] = TARGET_BASE_URL
os.environ['LOAD_TEST_PASSWORD'] = password
os.environ['LOAD_TEST_RUN_ID'] = 'kg_' + uuid.uuid4().hex[:12]
os.environ['LOAD_TEST_DATA'] = str(WORK / 'accounts.json')
# Chỉ gắn nhãn Kaggle khi thực sự chạy trong môi trường Kaggle.
os.environ['LOAD_TEST_ORIGIN'] = ('kaggle-cpu' if Path('/kaggle/working').is_dir()
                                and os.environ.get('KAGGLE_KERNEL_RUN_TYPE') else 'unverified-notebook')
metadata = {'backend': BACKEND_DESCRIPTION, 'generator_os': platform.platform(),
            'python': platform.python_version(), 'logical_cpus': os.cpu_count(),
            'memory_bytes': psutil.virtual_memory().total,
            'accelerator': 'CPU - confirm None in Kaggle settings',
            'origin': os.environ['LOAD_TEST_ORIGIN']}
(WORK / 'environment.json').write_text(json.dumps(metadata, indent=2))
with (WORK / 'pip-freeze.txt').open('w') as f:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=f, check=True)
print(metadata)


In [ ]:
import requests
# Chặn sớm lỗi mạng/target trước khi tạo accounts hoặc phát tải.
response = requests.get(TARGET_BASE_URL + '/api/health', timeout=10)
response.raise_for_status()
assert response.json() == {'status': 'UP'}
products = requests.get(TARGET_BASE_URL + '/api/products', timeout=10)
products.raise_for_status()
assert any(isinstance(p.get('stock'), int) and p['stock'] >= 2 for p in products.json()), 'Backend cần seed product stock >= 2'
print('HTTP preflight thành công; health không phải readiness DB/Redis.')


## Chuẩn bị và phát tải

Smoke trước; chỉ tăng sang low/medium/high khi có quyền và smoke đạt yêu cầu. Thay PROFILE rồi chạy lại hai cell dưới cho mỗi mức tải. Đây là tải có thời gian nghỉ 0.5–1.5s; không phải số request/s cố định. Prepare ngoài phép đo, login/relogin trong Locust được tính trong metrics.

In [ ]:
PROFILE = 'smoke'  # smoke / low / medium / high
profiles = {'smoke': 1, 'low': 10, 'medium': 50, 'high': 100}
subprocess.run([sys.executable, str(WORK / 'prepare.py'), '--users', str(profiles[PROFILE]),
                '--output', os.environ['LOAD_TEST_DATA']], check=True)


In [ ]:
from datetime import datetime, timezone
OUTPUT = WORK / 'results' / (PROFILE + '-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
result = subprocess.run([sys.executable, str(WORK / 'run.py'), '--profile', PROFILE,
                         '--output', str(OUTPUT)], check=False, timeout=3600)
print('Locust exit code:', result.returncode)
if (OUTPUT / 'summary.json').exists():
    summary = json.loads((OUTPUT / 'summary.json').read_text())
    print(json.dumps(summary, indent=2))
else:
    raise RuntimeError('Không có metrics hoàn chỉnh; xem locust.log')
if result.returncode:
    raise RuntimeError('Workload có lỗi. Giữ artifacts để điều tra, không báo PASS.')


## Phân tích dữ liệu thực

Chỉ vẽ khi đã có CSV. p95/p99 lấy từ histogram Locust, không trung bình percentile theo endpoint. Failure rate gồm cả HTTP lỗi và DTO mismatch. Độ trễ được quan sát từ Kaggle nên bao gồm mạng; không suy ra latency nội bộ backend. Cần dữ liệu CPU/RAM/DB/cache phía backend để xác định bottleneck.

In [ ]:
import csv
import matplotlib.pyplot as plt
with (OUTPUT / 'locust_stats_history.csv').open() as handle:
    history = list(csv.DictReader(handle))
aggregate = [r for r in history if r['Name'] == 'Aggregated']
if aggregate:
    start = float(aggregate[0]['Timestamp'])
    seconds = [float(r['Timestamp']) - start for r in aggregate]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(seconds, [float(r['Requests/s']) for r in aggregate])
    axes[0].set(xlabel='Thời gian (s)', ylabel='Requests/s', title='RPS quan sát')
    axes[1].plot(seconds, [float(r['User Count']) for r in aggregate])
    axes[1].set(xlabel='Thời gian (s)', ylabel='Virtual users', title='Concurrency quan sát')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'metrics.png', dpi=150)
    plt.show()
print('RPS:', summary['rps'], 'p95 ms:', summary['p95_latency_ms'],
      'p99 ms:', summary['p99_latency_ms'], 'error rate:', summary['failure_rate'])


In [ ]:
# So sánh chỉ các runs thực sự có summary, không tạo hàng số liệu giả.
comparison = []
for file in sorted((WORK / 'results').glob('*/summary.json')):
    data = json.loads(file.read_text())
    comparison.append({'run': file.parent.name, **{k: data[k] for k in
        ['measurement_origin', 'configured_users', 'peak_observed_users', 'total_requests',
         'rps', 'average_latency_ms', 'p95_latency_ms', 'p99_latency_ms', 'failure_rate', 'locust_exit_code']}})
if comparison:
    with (WORK / 'comparison.csv').open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
        writer.writeheader()
        writer.writerows(comparison)
    for row in comparison:
        print(row)
# Chỉ đóng gói báo cáo, không accounts/password/token. Rà soát target trong HTML/log trước chia sẻ.
EXPORT = WORK / 'export'
EXPORT.mkdir(exist_ok=True)
for name in ['environment.json', 'pip-freeze.txt', 'comparison.csv']:
    if (WORK / name).exists():
        shutil.copy2(WORK / name, EXPORT / name)
if (WORK / 'results').exists():
    shutil.copytree(WORK / 'results', EXPORT / 'results', dirs_exist_ok=True)
archive = shutil.make_archive(str(WORK / 'cart-load-report'), 'zip', EXPORT)
print('Báo cáo:', archive)


## Kết luận sau khi chạy

Tải ZIP/CSV/HTML về, ghi nhận revision, thời điểm, cấu hình CPU thực tế, môi trường backend, concurrency, RPS, average/p95/p99, failure rate và HTTP status distribution trong `docs/load-testing.md`. Phân tích lỗi trước khi kết luận bottleneck. Không coi smoke local hoặc notebook chưa chạy là benchmark Kaggle. Không tự động public notebook/output chứa địa chỉ nhạy cảm.